In [ ]:
from pathlib import Path
from typing import Optional, Any, Union

import numpy as np
import polars as pl
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State, callback_context
import plotly.graph_objects as go

# scipy
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# muutils
import muutils.tensor_info
from muutils.dbg import dbg, dbg_tensor

# attention-motifs
from attention_motifs.features.analysis import nan_stats, filter_data, normalize_data
from attention_motifs.features.plotting import (
	plot_correlation_matrix,
	plot_embedding,
	apply_pca,
)

muutils.tensor_info.DEFAULT_SETTINGS["colored"] = True
# pl.set_option('display.max_rows', 200)
# pl.set_option('display.max_columns', 200)

In [ ]:
# DATA: pl.DataFrame = pl.DataFrame(jsonl_load("../data/scalar_features.jsonl"))

DATA: pl.DataFrame = pl.read_ndjson(Path("../data/features/scalar_features_fixed.jsonl"))

In [ ]:
DATA.head()

In [ ]:
# print unique values for columns which start with "activations"
print(f"{DATA['activation.model'].unique() = }")

In [ ]:
for col in DATA.columns:
	if col.startswith("activation"):
		print(f"{col = }")

In [ ]:
DATA.describe()

In [ ]:
nan_stats(DATA)

In [ ]:
# because those have nan
DATA_FILTERED: pl.DataFrame = filter_data(
	DATA, remove_models=["tiny-stories-1M", "pythia-14m"]
)
nan_stats(DATA_FILTERED)

In [ ]:
# Pick feature columns
FEATURE_COLS: list[str] = [
	col for col in DATA_FILTERED.columns if col.startswith("feat.")
]

# Plot correlation matrix
plot_correlation_matrix(DATA_FILTERED, FEATURE_COLS)

In [ ]:
DATA_SCALED: pl.DataFrame = normalize_data(DATA_FILTERED, FEATURE_COLS)
DATA_SCALED

# plot_correlation_matrix(DATA_SCALED, FEATURE_COLS)

In [ ]:
# PCA
pca_data: np.ndarray
pca_obj: PCA
pca_data, pca_obj = apply_pca(DATA_SCALED, n_components=10, feature_cols=FEATURE_COLS)

In [ ]:
# print(dir(pca_obj))
# print(pca_obj._repr_html_())
display(pca_obj)

In [ ]:
# save some data
DATA_SCALED.write_ndjson(Path("../data/features/features_scaled.jsonl"))
dbg_tensor(pca_data)
np.save("../data/features/pca_data.npy", pca_data)

# only columns that start with "activation."
data_meta: pl.DataFrame = DATA_SCALED[[
	col for col in DATA_SCALED.columns if col.startswith("activation.")
]]
data_meta = data_meta.write_ndjson(Path("../data/features/features_meta.jsonl"))

In [ ]:
# for i in range(1, 3):
# 	for j in range(0, i):
# 		plot_embedding(
# 			pca_data,
# 			DATA_SCALED["activation.model"],
# 			(i, j),
# 			alpha=0.05,
# 			marker_size=10,
# 			title=f"2D PCA Embedding ({i}, {j})",
# 		)

In [ ]:
from attention_motifs.features.plotting import plot_embedding_kde
plot_embedding_kde(pca_data[:, :2], DATA_SCALED["activation.model"], "PCA of Features")

In [ ]:
def apply_tsne(
	data: pl.DataFrame, n_components: int = 2, perplexity: float = 30.0
) -> np.ndarray:
	"""Compute t-SNE."""
	tsne: TSNE = TSNE(n_components=n_components, perplexity=perplexity, random_state=0)
	embedded: np.ndarray = tsne.fit_transform(data)
	return embedded


def cluster_kmeans(data: np.ndarray, n_clusters: int) -> np.ndarray:
	"""Cluster with K-Means."""
	kmeans: KMeans = KMeans(n_clusters=n_clusters, random_state=0)
	labels: np.ndarray = kmeans.fit_predict(data)
	return labels


def plot_embedding(embedding: np.ndarray, labels: pl.Series, title: str) -> None:
	"""Scatter plot of 2D embedding with label text."""
	fig: plt.Figure = plt.figure()
	ax: plt.Axes = fig.add_subplot(111)
	sc: plt.PathCollection = ax.scatter(embedding[:, 0], embedding[:, 1])
	ax.set_title(title)
	ax.set_aspect("equal")
	ax.set_xlabel("Dim 1")
	ax.set_ylabel("Dim 2")
	# Optionally, overlay text for each point
	# for i, lbl in enumerate(labels):
	#     ax.text(embedding[i, 0], embedding[i, 1], lbl, fontsize=6)
	plt.show()


def main_example(df: pl.DataFrame) -> None:
	# t-SNE
	tsne_data: np.ndarray = apply_tsne(df)

	# Clustering
	kmeans_labels: np.ndarray = cluster_kmeans(tsne_data, n_clusters=5)

	# Plot PCA embedding
	# (using first two PCA components for a 2D plot)
	plot_embedding(pca_data[:, :2], df["activation.cls"], "PCA of Features")

	# Plot t-SNE embedding
	plot_embedding(tsne_data, df["activation.cls"], "t-SNE of Features")

	# Plot the clusters on t-SNE
	# (If you prefer numeric cluster labels, you can cast them to string.)
	cluster_series: pl.Series = pl.Series(kmeans_labels, index=df.index, dtype=str)
	plot_embedding(tsne_data, cluster_series, "K-Means Clusters (t-SNE)")


# main_example(DATA_NANLESS)